# Evaluación de reconstrucciones VAE / DDSP

## 0. Instalación de dependencias

Descomenta la siguiente línea si te falta alguna librería.

In [89]:
# pip install numpy librosa mir_eval pandas matplotlib soundfile

In [90]:
import os
import numpy as np
import librosa
import pandas as pd
import matplotlib.pyplot as plt
import mir_eval.melody as melody

## 1. Configuración — pon aquí tus rutas

- `ORIGINALS_DIR`: carpeta con los audios originales (ground truth).
- `RECONSTRUCTIONS_DIR`: carpeta con los audios reconstruidos por tu VAE/DDSP.
- Los archivos se emparejan por **nombre de fichero idéntico** en ambas carpetas (p.ej. `originals/nota_01.wav` con `reconstructions/nota_01.wav`).
- Si las rutas no existen o no hay archivos emparejados, el notebook genera un par sintético de ejemplo para que puedas comprobar que todo funciona antes de apuntar a tus datos reales.

In [91]:
ORIGINALS_DIR = "/mnt/wsl/PHYSICALDRIVE1p2/home/antipersona/PROYECTOS/TFG-MUSICAL/examples/Comparations/AE/og/"
RECONSTRUCTIONS_DIR = "/mnt/wsl/PHYSICALDRIVE1p2/home/antipersona/PROYECTOS/TFG-MUSICAL/examples/Comparations/AE/GL/"
SR = 16000
OUTPUT_CSV = "resultados_evaluacion_ae_pr.csv"

## 2. Carga de audios

In [92]:
def find_paired_files(originals_dir, reconstructions_dir):
    """Empareja archivos con el mismo nombre en ambas carpetas."""
    if not (os.path.isdir(originals_dir) and os.path.isdir(reconstructions_dir)):
        return []
    exts = (".wav", ".flac", ".mp3", ".ogg")
    orig_files = {f for f in os.listdir(originals_dir) if f.lower().endswith(exts)}
    rec_files = {f for f in os.listdir(reconstructions_dir) if f.lower().endswith(exts)}
    common = sorted(orig_files & rec_files)
    if not common:
        print(f"Aviso: no hay archivos con el mismo nombre en ambas carpetas ({len(orig_files)} en originals, {len(rec_files)} en reconstructions).")
    return [(os.path.join(originals_dir, f), os.path.join(reconstructions_dir, f)) for f in common]


def load_pairs_from_dirs(originals_dir, reconstructions_dir, sr):
    pairs = find_paired_files(originals_dir, reconstructions_dir)
    loaded = []
    for orig_path, rec_path in pairs:
        name = os.path.basename(orig_path)
        y_orig, _ = librosa.load(orig_path, sr=sr, mono=True)
        y_rec, _ = librosa.load(rec_path, sr=sr, mono=True)
        loaded.append((name, y_orig, y_rec))
    return loaded

In [ ]:
audio_pairs = load_pairs_from_dirs(ORIGINALS_DIR, RECONSTRUCTIONS_DIR, SR)
print(f"{len(audio_pairs)} par(es) de audio cargado(s): {[name for name, _, _ in audio_pairs]}")

4 par(es) de audio cargado(s): ['1.wav', '2.wav', '3.wav', '4.wav']


## 3. Métricas de reconstrucción

MSE y MAE calculados sobre el espectrograma de magnitud, tal y como se definen en Vinay & Lerch (2022). El *multi-scale spectral loss* (Engel et al., DDSP) se incluye como referencia adicional — **no** es una de las 8 métricas de la Tabla 1 del paper de Vinay & Lerch, pero sí una métrica de reconstrucción muy usada en la literatura DDSP, así que se mantiene aparte para no confundir la atribución.

In [94]:
def _align_length(a, b):
    n = min(len(a), len(b))
    return a[:n], b[:n]


def spectrogram_mse_mae(original, reconstructed, n_fft=1024, hop_length=256):
    original, reconstructed = _align_length(original, reconstructed)
    S_o = np.abs(librosa.stft(original, n_fft=n_fft, hop_length=hop_length))
    S_r = np.abs(librosa.stft(reconstructed, n_fft=n_fft, hop_length=hop_length))
    n = min(S_o.shape[1], S_r.shape[1])
    S_o, S_r = S_o[:, :n], S_r[:, :n]
    return {"MSE": float(np.mean((S_o - S_r) ** 2)), "MAE": float(np.mean(np.abs(S_o - S_r)))}


def multiscale_spectral_loss(original, reconstructed, fft_sizes=(2048, 1024, 512, 256, 128, 64)):
    original, reconstructed = _align_length(original, reconstructed)
    loss = 0.0
    for n_fft in fft_sizes:
        hop = max(n_fft // 4, 1)
        S_o = np.abs(librosa.stft(original, n_fft=n_fft, hop_length=hop))
        S_r = np.abs(librosa.stft(reconstructed, n_fft=n_fft, hop_length=hop))
        n = min(S_o.shape[1], S_r.shape[1])
        S_o, S_r = S_o[:, :n], S_r[:, :n]
        loss += np.mean(np.abs(S_o - S_r)) + np.mean(np.abs(np.log(S_o + 1e-8) - np.log(S_r + 1e-8)))
    return float(loss / len(fft_sizes))


def compare_audio_reconstruction(original, reconstructed, n_fft=1024, hop_length=256):
    metrics = spectrogram_mse_mae(original, reconstructed, n_fft, hop_length)
    metrics["MSL"] = multiscale_spectral_loss(original, reconstructed)
    return metrics

## 4. Predicción de descriptores acústicos — f0 y loudness (sección 4.3.2)

- **f0**: extraído con pYIN; se compara con las métricas estándar de `mir_eval.melody` (RPA, RCA, Overall Accuracy, Voicing Recall/False Alarm) más el RMSE en cents.
- **Loudness**: energía A-weighted del espectro en dB; se compara con MAE, RMSE y correlación de Pearson.

In [95]:
def extract_f0(audio, sr, fmin=50.0, fmax=2000.0, frame_length=1024, hop_length=256):
    f0, voiced_flag, _ = librosa.pyin(
        audio, fmin=fmin, fmax=fmax, sr=sr,
        frame_length=frame_length, hop_length=hop_length,
    )
    f0 = np.nan_to_num(f0, nan=0.0)
    times = librosa.times_like(f0, sr=sr, hop_length=hop_length)
    return f0, voiced_flag.astype(bool), times


def extract_loudness(audio, sr, n_fft=1024, hop_length=256):
    S = np.abs(librosa.stft(audio, n_fft=n_fft, hop_length=hop_length))
    freqs = librosa.fft_frequencies(sr=sr, n_fft=n_fft)
    a_weighting_db = librosa.A_weighting(freqs)
    power = (S ** 2) * (10 ** (a_weighting_db[:, None] / 10))
    return 10 * np.log10(np.mean(power, axis=0) + 1e-10)

In [96]:
def compare_f0(original, reconstructed, sr, fmin=50.0, fmax=2000.0, hop_length=256, cent_tolerance=50.0):
    f0_o, _, times_o = extract_f0(original, sr, fmin, fmax, hop_length=hop_length)
    f0_r, _, _ = extract_f0(reconstructed, sr, fmin, fmax, hop_length=hop_length)
    n = min(len(f0_o), len(f0_r))
    f0_o, f0_r = f0_o[:n], f0_r[:n]
    times = times_o[:n]

    ref_freq = np.where(f0_o > 0, f0_o, 0.0)
    est_freq = np.where(f0_r > 0, f0_r, 0.0)

    scores = melody.evaluate(times, ref_freq, times, est_freq, cent_tolerance=cent_tolerance)

    both_voiced = (ref_freq > 0) & (est_freq > 0)
    if np.any(both_voiced):
        cents_diff = 1200 * np.log2(est_freq[both_voiced] / ref_freq[both_voiced])
        f0_rmse_cents = float(np.sqrt(np.mean(cents_diff ** 2)))
    else:
        f0_rmse_cents = float("nan")

    return {
        "Voicing Recall": float(scores["Voicing Recall"]),
        "Voicing False Alarm": float(scores["Voicing False Alarm"]),
        "RPA": float(scores["Raw Pitch Accuracy"]),
        "RCA": float(scores["Raw Chroma Accuracy"]),
        "OA": float(scores["Overall Accuracy"]),
        "f0 RMSE (cents)": f0_rmse_cents,
    }


def compare_loudness(original, reconstructed, sr, n_fft=1024, hop_length=256):
    l_o = extract_loudness(original, sr, n_fft, hop_length)
    l_r = extract_loudness(reconstructed, sr, n_fft, hop_length)
    n = min(len(l_o), len(l_r))
    l_o, l_r = l_o[:n], l_r[:n]
    return {
        "Loud. MAE (dB)": float(np.mean(np.abs(l_o - l_r))),
        "Loud. RMSE (dB)": float(np.sqrt(np.mean((l_o - l_r) ** 2))),
        "Loud. Corr.": float(np.corrcoef(l_o, l_r)[0, 1]) if n > 1 else float("nan"),
    }


def compare_acoustic_descriptors(original, reconstructed, sr, **kwargs):
    metrics = compare_f0(original, reconstructed, sr)
    metrics.update(compare_loudness(original, reconstructed, sr))
    return metrics

## 5. Ejecutar todo sobre tus audios

In [97]:
rows = []
for name, y_orig, y_rec in audio_pairs:
    row = {"file": name}
    row.update(compare_audio_reconstruction(y_orig, y_rec))
    row.update(compare_acoustic_descriptors(y_orig, y_rec, SR))
    rows.append(row)

results_df = pd.DataFrame(rows).set_index("file")
results_df

/home/antipersona/.venvs/AUDIO/lib/python3.13/site-packages/librosa/core/convert.py:1869: RuntimeWarning: divide by zero encountered in log10
  + 2 * np.log10(f_sq)


,MSE,MAE,MSL,Voicing Recall,Voicing False Alarm,RPA,RCA,OA,f0 RMSE (cents),Loud. MAE (dB),Loud. RMSE (dB),Loud. Corr.
file,,,,,,,,,,,,
1.wav,4.936826,0.258799,1.162557,1.000000,0.073171,0.969072,0.969072,0.961702,23.375024,5.368884,8.123738,0.977017
2.wav,14.861906,0.320404,2.875002,0.890411,0.000000,0.867580,0.867580,0.876596,19.268681,12.106020,21.673340,0.874432
3.wav,9.697536,0.487142,1.485396,0.974619,0.052632,0.934010,0.934010,0.936170,24.906074,2.582781,6.609575,0.987189
4.wav,12.294744,0.625227,4.366590,0.948357,0.000000,0.906103,0.906103,0.914894,21.387514,7.663882,13.927744,0.980221


## 6. Resultados agregados

Media y desviación típica por métrica — los números que puedes citar directamente en la tabla de resultados del TFG.

In [98]:
summary = results_df.agg(["mean", "std"])
summary

,MSE,MAE,MSL,Voicing Recall,Voicing False Alarm,RPA,RCA,OA,f0 RMSE (cents),Loud. MAE (dB),Loud. RMSE (dB),Loud. Corr.
mean,10.447753,0.422893,2.472386,0.953347,0.031451,0.919191,0.919191,0.922340,22.234323,6.930392,12.583599,0.954715
std,4.235928,0.165829,1.465140,0.046957,0.037271,0.042983,0.042983,0.036003,2.446192,4.027620,6.831531,0.053690


In [99]:
results_df.to_csv(OUTPUT_CSV)
print(f"Resultados por archivo guardados en: {OUTPUT_CSV}")

Resultados por archivo guardados en: resultados_evaluacion_ae_pr.csv


## 7. (Opcional) Visualizar f0 y loudness de un ejemplo

Compara visualmente las curvas de f0 y loudness del primer par original/reconstruido

In [100]:
# name0, y_orig0, y_rec0 = audio_pairs[0]

# f0_o, _, times_o = extract_f0(y_orig0, SR)
# f0_r, _, times_r = extract_f0(y_rec0, SR)
# loud_o = extract_loudness(y_orig0, SR)
# loud_r = extract_loudness(y_rec0, SR)

# fig, axes = plt.subplots(2, 1, figsize=(10, 6))

# axes[0].plot(times_o, f0_o, label="Original")
# axes[0].plot(times_r[:len(f0_r)], f0_r, label="Reconstructed", alpha=0.8)
# axes[0].set_ylabel("f0 (Hz)")
# axes[0].set_title(f"f0 — {name0}")
# axes[0].legend()

# t_loud = librosa.times_like(loud_o, sr=SR, hop_length=256)
# axes[1].plot(t_loud, loud_o, label="Original")
# axes[1].plot(t_loud[:len(loud_r)], loud_r, label="Reconstructed", alpha=0.8)
# axes[1].set_ylabel("Loudness (dB)")
# axes[1].set_xlabel("Tiempo (s)")
# axes[1].set_title(f"Loudness — {name0}")
# axes[1].legend()

# plt.tight_layout()
# plt.show()

## 8. Fréchet Audio Distance (FAD)

FAD (Kilgour et al., 2019) es la métrica de distancia distribucional que el paper de Vinay & Lerch cita para DiffWave, Neural Waveshaping Synthesis, DarkGAN y CRASH, y es la que mejor separa a los sistemas en su Tabla 1.

A diferencia de MSE/MAE, **no se calcula por pares**: se ajusta una gaussiana multivariante a los embeddings VGGish de *todo* el conjunto de originales y otra a los de *todo* el conjunto de reconstrucciones, y se mide la distancia de Fréchet entre esas dos distribuciones. Un valor más bajo indica que las dos distribuciones (y por tanto el "timbre global" que producen) están más cerca; 0 sería reconstrucción distribucionalmente idéntica.

Como es una métrica sobre una *población* de sonidos y no sobre un único archivo, hacen falta varios archivos por carpeta — con el par sintético de demo (1 archivo) no se puede calcular de forma significativa.

In [101]:
# pip install frechet-audio-distance --break-system-packages
# (usa embeddings VGGish vía PyTorch — el mismo backbone que emplea el paper)

In [102]:
SYSTEM_NAME = "VAE"  # nombre de este sistema/checkpoint; cámbialo cuando evalúes otro modelo


def _import_frechet_audio_distance():
    """Importa FrechetAudioDistance evitando el crash de laion_clap.

    frechet_audio_distance importa internamente laion_clap (aunque solo uses VGGish),
    y laion_clap llama a argparse.parse_args() al importarse, leyendo sys.argv. Dentro
    de Jupyter, sys.argv trae los argumentos del propio kernel (--f=.../kernel-....json),
    que no coinciden con ningún argumento esperado -> "ambiguous option". Se vacía
    sys.argv temporalmente durante el import para evitarlo.
    """
    import sys

    argv_original = sys.argv
    try:
        sys.argv = [sys.argv[0]]
        from frechet_audio_distance import FrechetAudioDistance
    finally:
        sys.argv = argv_original
    return FrechetAudioDistance


import contextlib


@contextlib.contextmanager
def _sqrtm_disp_compat():
    """Compatibilidad con scipy reciente, donde linalg.sqrtm() ya no acepta 'disp'.

    frechet_audio_distance calcula la distancia de Fréchet con el boilerplate clásico
    de FID: `covmean, _ = linalg.sqrtm(sigma1 @ sigma2, disp=False)`, esperando una
    tupla (matriz, error_estimado). En versiones recientes de scipy ese parámetro se
    ha eliminado y sqrtm() siempre devuelve solo la matriz, lo que provoca el
    "An error occurred: sqrtm() got an unexpected keyword argument 'disp'" y hace que
    la librería devuelva -1.0 como FAD en vez de un valor real. Este parche envuelve
    scipy.linalg.sqrtm solo durante el cálculo del FAD, y lo restaura después.
    """
    from scipy import linalg as scipy_linalg

    sqrtm_original = scipy_linalg.sqrtm

    def sqrtm_compat(A, disp=True, blocksize=64):
        try:
            return sqrtm_original(A, disp=disp, blocksize=blocksize)
        except TypeError:
            resultado = sqrtm_original(A)
            if disp:
                return resultado
            error_estimado = np.linalg.norm(resultado.dot(resultado) - A, "fro") / (
                np.linalg.norm(A, "fro") + 1e-12
            )
            return resultado, error_estimado

    scipy_linalg.sqrtm = sqrtm_compat
    try:
        yield
    finally:
        scipy_linalg.sqrtm = sqrtm_original


def compute_fad(originals_dir, reconstructions_dir, sr=16000):
    """FAD entre la distribución de originales y la de reconstrucciones (embeddings VGGish)."""
    try:
        FrechetAudioDistance = _import_frechet_audio_distance()
    except (ImportError, SystemExit):
        print("Falta la librería: pip install frechet-audio-distance --break-system-packages")
        return float("nan")

    if not (os.path.isdir(originals_dir) and os.path.isdir(reconstructions_dir)):
        print("FAD requiere carpetas reales con varios archivos; no se calcula sobre el par sintético de demo.")
        return float("nan")

    frechet = FrechetAudioDistance(
        model_name="vggish",
        sample_rate=sr,
        use_pca=False,
        use_activation=False,
        verbose=False,
    )
    with _sqrtm_disp_compat():
        return float(frechet.score(originals_dir, reconstructions_dir))


fad_score = compute_fad(ORIGINALS_DIR, RECONSTRUCTIONS_DIR, SR)
print(f"FAD ({SYSTEM_NAME}): {fad_score:.4f}")

Using cache found in /home/antipersona/.cache/torch/hub/harritaylor_torchvggish_master


FAD (VAE): 8.5485


## 9. Tabla comparativa entre sistemas (estilo Tabla 1 del paper)

El paper compara varios sistemas en una única tabla (DDSP, DiffWave, NSynth, Anchor). Esta celda guarda un CSV persistente (`comparativa_sistemas.csv`): cada vez que ejecutes el notebook con un `RECONSTRUCTIONS_DIR` distinto (otro modelo, otro checkpoint, un *anchor* degradado a propósito, etc.) y actualices `SYSTEM_NAME`, se añade o actualiza la fila correspondiente, de modo que vas construyendo una tabla comparativa multisistema en vez de resultados sueltos por ejecución.

In [103]:
# COMPARATIVA_CSV = "comparativa_sistemas.csv"

# fila_resumen = summary.loc["mean"].to_dict()
# fila_resumen["fad"] = fad_score

# if os.path.exists(COMPARATIVA_CSV):
#     comparativa = pd.read_csv(COMPARATIVA_CSV, index_col=0)
# else:
#     comparativa = pd.DataFrame()

# comparativa.loc[SYSTEM_NAME] = fila_resumen
# comparativa.to_csv(COMPARATIVA_CSV)
# comparativa

## 10. Exportar tablas a LaTeX

Dos tablas listas para pegar en la memoria del TFG:

1. **Resultados por archivo** (`tabla_resultados_por_archivo.tex`) — para un apéndice.
2. **Comparativa entre sistemas** (`tabla_comparativa_sistemas.tex`) — con flechas ↓/↑ de "mejor dirección" por columna y el mejor valor de cada métrica en negrita, igual que la Tabla 1 del paper. Solo usa `\usepackage{booktabs}` en el preámbulo de LaTeX.

In [104]:
def _escapar_latex(texto):
    return str(texto).replace("_", "\\_")


def _dividir_en_grupos(elementos, tam_max=6):
    """Divide una lista en grupos de como mucho tam_max elementos."""
    return [elementos[i:i + tam_max] for i in range(0, len(elementos), tam_max)]


def guardar_tabla_latex(df, path, caption, label, float_format="%.4f", max_cols=6):
    """Vuelca un DataFrame a una o varias tablas LaTeX (tabular + table + caption + label).

    Si el DataFrame tiene más de max_cols columnas, se reparte en varias tablas de como
    mucho max_cols columnas cada una (misma filas, distinta caption/label numerada), para
    que quepan en la página. El texto va en \\footnotesize.
    """
    df_export = df.rename(columns=_escapar_latex)
    df_export.index = df_export.index.map(_escapar_latex)

    grupos_columnas = _dividir_en_grupos(list(df_export.columns), max_cols)
    n_grupos = len(grupos_columnas)

    bloques = []
    for i, columnas in enumerate(grupos_columnas, start=1):
        cuerpo_tabular = df_export[columnas].to_latex(float_format=float_format, escape=False)
        sufijo_caption = f" ({i}/{n_grupos})" if n_grupos > 1 else ""
        sufijo_label = f"_{i}" if n_grupos > 1 else ""
        bloque = (
            "\\begin{table}[htbp]\n\\centering\n\\footnotesize\n"
            + cuerpo_tabular
            + f"\\caption{{{caption}{sufijo_caption}}}\n\\label{{{label}{sufijo_label}}}\n\\end{{table}}\n"
        )
        bloques.append(bloque)

    latex_str = "\n".join(bloques)
    with open(path, "w", encoding="utf-8") as f:
        f.write(latex_str)
    print(f"Tabla LaTeX guardada en: {path} ({n_grupos} tabla(s))")
    return latex_str


# Dirección de "mejor valor" por métrica (↓ = menor es mejor, ↑ = mayor es mejor)
DIRECCION_METRICAS = {
    "mse": "min", "mae": "min", "multiscale_spectral_loss": "min", "fad": "min",
    "voicing_recall": "max", "voicing_false_alarm": "min",
    "raw_pitch_accuracy": "max", "raw_chroma_accuracy": "max", "overall_accuracy": "max",
    "f0_rmse_cents": "min", "loudness_mae_db": "min", "loudness_rmse_db": "min",
    "loudness_correlation": "max",
}

# Abreviaturas para que los encabezados ocupen mucho menos espacio
ABREVIATURAS_METRICAS = {
    "mse": "MSE",
    "mae": "MAE",
    "multiscale_spectral_loss": "MSL",
    "fad": "FAD",
    "voicing_recall": "VR",
    "voicing_false_alarm": "VFA",
    "raw_pitch_accuracy": "RPA",
    "raw_chroma_accuracy": "RCA",
    "overall_accuracy": "OA",
    "f0_rmse_cents": "F0-RMSE",
    "loudness_mae_db": "L-MAE",
    "loudness_rmse_db": "L-RMSE",
    "loudness_correlation": "L-Corr",
}


def _escapar_latex(texto):
    return str(texto).replace("_", r"\_")


def _dividir_en_grupos(elementos, tam_max=6):
    """Divide una lista en grupos de como mucho tam_max elementos."""
    return [
        elementos[i:i + tam_max]
        for i in range(0, len(elementos), tam_max)
    ]


def formatear_valor_latex(valor, es_mejor, decimales=3):
    """Formatea un valor y pone en negrita el mejor."""
    texto = f"{valor:.{decimales}f}"
    return rf"\textbf{{{texto}}}" if es_mejor else texto


def tabla_comparativa_a_latex(
    df,
    direcciones,
    path,
    caption,
    label,
    max_cols=8,
    decimales=3,
    abreviaturas=None,
):
    """
    Genera una tabla LaTeX compacta para un paper.

    Características:
    - Encabezados abreviados.
    - Fuente \\scriptsize.
    - Poco espacio horizontal entre columnas.
    - Poco espacio vertical entre filas.
    - Flechas ↑/↓ para indicar si mayor/menor es mejor.
    - Mejor resultado en negrita.
    - Divide automáticamente en varias tablas si hay demasiadas métricas.

    Parámetros
    ----------
    df : pd.DataFrame
        DataFrame con sistemas en filas y métricas en columnas.

    direcciones : dict
        Diccionario indicando "min" o "max" para cada métrica.

    path : str
        Archivo .tex de salida.

    caption : str
        Caption de la tabla.

    label : str
        Label de LaTeX.

    max_cols : int
        Máximo número de métricas por tabla.

    decimales : int
        Número de decimales mostrados.

    abreviaturas : dict, opcional
        Diccionario {nombre_original: nombre_abreviado}.
    """

    if abreviaturas is None:
        abreviaturas = ABREVIATURAS_METRICAS

    # Solo métricas que estén en direcciones
    columnas = [
        c for c in df.columns
        if c in direcciones
    ]

    # Mejor sistema para cada métrica
    mejores = {}

    for col in columnas:
        if direcciones[col] == "max":
            mejores[col] = df[col].idxmax()
        elif direcciones[col] == "min":
            mejores[col] = df[col].idxmin()
        else:
            raise ValueError(
                f"Dirección inválida para '{col}': "
                f"{direcciones[col]}. Usa 'min' o 'max'."
            )

    flechas = {
        "max": r"$\uparrow$",
        "min": r"$\downarrow$",
    }

    # Dividir las métricas en grupos
    grupos_columnas = _dividir_en_grupos(columnas, max_cols)
    n_grupos = len(grupos_columnas)

    bloques = []

    for i, cols_grupo in enumerate(grupos_columnas, start=1):

        # ---------------------------------------------------------
        # Encabezados abreviados
        # ---------------------------------------------------------
        encabezados = []

        for col in cols_grupo:
            nombre_corto = abreviaturas.get(col, col)
            nombre_corto = _escapar_latex(nombre_corto)

            encabezados.append(
                f"{nombre_corto} {flechas[direcciones[col]]}"
            )

        encabezado = " & ".join(encabezados)

        # ---------------------------------------------------------
        # Comienzo de la tabla
        # ---------------------------------------------------------
        lineas = [
            r"\begin{table}[!t]",
            r"\centering",
            r"\scriptsize",
            r"\setlength{\tabcolsep}{3pt}",
            r"\renewcommand{\arraystretch}{0.85}",
            r"\begin{tabular}{@{}l" + "c" * len(cols_grupo) + "@{}}",
            r"\toprule",
            f"Sistema & {encabezado} \\\\",
            r"\midrule",
        ]

        # ---------------------------------------------------------
        # Filas
        # ---------------------------------------------------------
        for sistema, fila in df.iterrows():

            valores = []

            for col in cols_grupo:
                valor = fila[col]

                texto = formatear_valor_latex(
                    valor,
                    mejores[col] == sistema,
                    decimales=decimales,
                )

                valores.append(texto)

            fila_latex = (
                f"{_escapar_latex(sistema)} & "
                + " & ".join(valores)
                + r" \\"
            )

            lineas.append(fila_latex)

        # ---------------------------------------------------------
        # Final de tabla
        # ---------------------------------------------------------
        sufijo_caption = (
            f" ({i}/{n_grupos})"
            if n_grupos > 1
            else ""
        )

        sufijo_label = (
            f"_{i}"
            if n_grupos > 1
            else ""
        )

        lineas += [
            r"\bottomrule",
            r"\end{tabular}",
            f"\\caption{{{caption}{sufijo_caption}}}",
            f"\\label{{{label}{sufijo_label}}}",
            r"\end{table}",
        ]

        bloques.append("\n".join(lineas))

    # Unir las tablas
    latex_str = "\n\n".join(bloques)

    # Guardar
    with open(path, "w", encoding="utf-8") as f:
        f.write(latex_str)

    print(
        f"Tabla LaTeX guardada en: {path} "
        f"({n_grupos} tabla(s))"
    )

    return latex_str

def formatear_valor_latex(valor, es_mejor, decimales=4):
    texto = f"{valor:.{decimales}f}"
    return f"\\textbf{{{texto}}}" if es_mejor else texto



In [105]:
# Tabla con el detalle por archivo (útil para un apéndice)
tex_por_archivo = guardar_tabla_latex(
    results_df,
    path="tabla_resultados_por_archivo.tex",
    caption="VAE reconstruction results",
    label="tab:resultados_por_archivo_VAE",
)

# # Tabla comparativa entre sistemas, estilo Tabla 1 del paper
# tex_comparativa = tabla_comparativa_a_latex(
#     comparativa,
#     DIRECCION_METRICAS,
#     path="tabla_comparativa_sistemas.tex",
#     caption="Comparativa de sistemas evaluados.",
#     label="tab:comparativa_sistemas",
# )

# print(tex_comparativa)

Tabla LaTeX guardada en: tabla_resultados_por_archivo.tex (2 tabla(s))
